# [nanobind](https://nanobind.readthedocs.io)

Same author as pybind11 (Wenzel Jakob), same style of code, written a decade later with the mistakes removed.

* Roughly 4x smaller binaries and 4x faster compiles than pybind11
* Requires C++17 and Python 3.8+
* Supports the Stable ABI (one wheel for many Pythons) and the free-threaded build
* `nb::ndarray` talks to NumPy, PyTorch, JAX, and anything else supporting DLPack

Downsides:

* CMake only; there is no setuptools path
* No embedding, and fewer conversions built in than pybind11
* You give up some flexibility: nanobind holds objects by value or by pointer, not by arbitrary smart pointer

If you are starting a new C++ binding today, this is usually the one to reach for. Existing pybind11 code is not automatically portable, but it looks close enough that porting is mostly mechanical.

In [ ]:
import os
import subprocess
import sys

# On macOS, point every compiler at the installed SDK
if sys.platform.startswith("darwin"):
    sdk = subprocess.run(
        ["xcrun", "--show-sdk-path"], capture_output=True, text=True, check=True
    )
    os.environ.setdefault("SDKROOT", sdk.stdout.strip())

Our usual example class:

In [ ]:
%%writefile SimpleClass.hpp
#pragma once

class Simple {
    int x;

public:

    Simple(int x): x(x) {}

    int get() const {
        return x;
    }

};

The binding code is almost the pybind11 code with the names changed: `py::` becomes `nb::`, and `PYBIND11_MODULE` becomes `NB_MODULE`.

In [ ]:
%%writefile nanoclass.cpp

#include <nanobind/nanobind.h>
#include "SimpleClass.hpp"

namespace nb = nanobind;

NB_MODULE(nanoclass, m) {
    nb::class_<Simple>(m, "Simple")
        .def(nb::init<int>())
        .def("get", &Simple::get)
    ;
}

# Building

nanobind is CMake-only. `nanobind_add_module` compiles the module and, unlike pybind11, also compiles a small support library it needs.

`STABLE_ABI` asks for an `abi3` module when the Python version supports it, and `NB_STATIC` links nanobind's support code into this one module.

In [ ]:
%%writefile CMakeLists.txt

cmake_minimum_required(VERSION 3.15...4.0)
project(nanoclass LANGUAGES CXX)

find_package(Python 3.11 REQUIRED COMPONENTS Interpreter Development.Module)
find_package(nanobind CONFIG REQUIRED)

nanobind_add_module(nanoclass STABLE_ABI NB_STATIC nanoclass.cpp)

install(TARGETS nanoclass DESTINATION .)

We saw scikit-build-core in the Minuit2 notebook; the same six lines make this project pip installable.

In [ ]:
%%writefile pyproject.toml

[build-system]
requires = ["scikit-build-core>=0.10", "nanobind>=2"]
build-backend = "scikit_build_core.build"

[project]
name = "nanoclass"
version = "0.1"
requires-python = ">=3.11"

[tool.scikit-build]
cmake.version = ">=3.15"

In [ ]:
!{sys.executable} -m pip install --no-build-isolation .

In [ ]:
import nanoclass

x = nanoclass.Simple(4)
x.get()

# What you get for free

Look at the error message when the argument type is wrong; nanobind writes the signature out for you.

In [ ]:
try:
    nanoclass.Simple("not an int")
except TypeError as e:
    print(e)

# Arrays

The feature most people come for is `nb::ndarray`, which accepts and returns arrays from any framework that speaks DLPack, without copying:

```cpp
#include <nanobind/ndarray.h>

void scale(nb::ndarray<double, nb::ndim<1>, nb::c_contig> a, double factor) {
    for (size_t i = 0; i < a.shape(0); i++)
        a(i) *= factor;
}
```

The same function then works on a NumPy array, a PyTorch tensor, or a JAX array, on CPU or GPU.

Pybind11's `py::array_t` only speaks NumPy.